In [1]:
import os
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

In [11]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [3]:

def get_products(prompt):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_embedding`,
          (SELECT @prompt AS content),  -- Aquí usamos el parámetro
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      ML.DISTANCE(
        qe.query_embedding,
        e.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_final_temp` AS d
    INNER JOIN
      `dataton-2024-team-01-cofares.datos_cofares.SalidaEmbeddings_temp` AS e
      ON d.codigo_web = e.title
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """.format(prompt)
    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:

        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'


        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')
        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query
        })
    return products

In [4]:
def rerank_products(prompt, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=product["descripcion"] + " " + product["modo_implementacion"]
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=prompt,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"]
        }
        for record in response.records[:5] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [5]:
#FUNCTION CALLING

# Define el schema
product_schema = FunctionDeclaration(
    name="product_query",
    description="Fetches relevant product information based on a search prompt.",
    parameters={
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "codigo_web": {"type": "string", "description": "Product web code"},
                        "nombre": {"type": "string", "description": "Product name"},
                        "codigo_nacional": {"type": "string", "description": "National product code"},
                        "descripcion": {"type": "string", "description": "Product description"},
                        "modo_implementacion": {"type": "string", "description": "Mode of implementation"},
                        "imagen_url": {"type": "string", "description": "Image URL"},
                        "distance_to_query": {"type": "number", "description": "Semantic distance to query"}
                    }
                }
            }
        }
    }
)

# Define tools antes de inicializar el modelo
tools = [Tool(function_declarations=[product_schema])]

In [6]:
PROJECT_ID = "dataton-2024-team-01-cofares"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

# Importa el modelo de Gemini Flash 1.5
import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)


# Model definition
multimodal_model = genai.GenerativeModel(
"gemini-1.5-flash",
generation_config=GenerationConfig(temperature=0),
tools=tools)

chat = multimodal_model.start_chat(response_validation=False)

In [7]:
def generate_response(prompt):  # Eliminamos el parámetro products
    #chat = multimodal_model.start_chat()

    instruction_prompt = f"""

    Eres una asistente farmacéutica experta llamada Cofarma.
    Tu tarea es responder de manera concisa y precisa a las consultas de los profesionales de la farmacia. 

    Cuando se te presente una consulta, deberás:
    - Entender la pregunta: Identifica claramente lo que el usuario está buscando.
    - Si se trata de un saludo, presentate y pregúntale en qué puedes ayudarles hoy. Ejemplos de saludos: "hola", "Hi", "Que tal?", "Como estas?".
    - Si la entrada solicita búsquedas no relacionadas con productos de farmacia, aclare que ese no es su propósito como asistente de búsqueda de productos de farmacia. Ejemplos de solicitudes no pertinentes: «Quiero la receta de una lasaña», “Quiero pedir una pizza”, “¿Qué tiempo hace hoy?”.
    - Buscar productos relevantes: Si la pregunta se relaciona con productos farmacéuticos, utiliza ${tools} para encontrar las opciones más adecuadas.
    - Si no encuentras productos relevantes, indica al usuario que no hay opciones disponibles.

    Aquí está la consulta del experto farmacéutico: {prompt}
    """

    try:
        response = chat.send_message(instruction_prompt)
        response.candidates[0].content.parts[0]
        
        # Verificar si hay una llamada a función
        for candidate in response.candidates:
            for part in candidate.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    # Ejecutar búsqueda de productos
                    products = get_products(prompt)
                    if not products:
                        return "Lo siento, no encontré productos que coincidan con tu búsqueda."
                    
                    ranked_products = rerank_products(prompt, products)
                    
                    # Enviar los resultados al modelo para generar una respuesta contextual
                    results_prompt = f"""
                    Basado en la búsqueda "{prompt}", he encontrado estos productos:
                    {[product['nombre'] for product in ranked_products['products']]}
                    
                    Por favor, genera una respuesta útil que:
                    1. Mencione los productos encontrados
                    2. Explique por qué son relevantes
                    3. Proporcione recomendaciones de uso
                    """
                    
                    final_response = chat.send_message(results_prompt)
                    return {
                        "type": "product_search",
                        "message": final_response.text,
                        "products": ranked_products["products"]
                    }
                
        # Si no hay llamada a función, devolver la respuesta conversacional
        return {
            "type": "conversation",
            "message": response.text
        }
                    
    except Exception as e:
        return f"Lo siento, ocurrió un error: {str(e)}"

In [20]:
# Ejemplo de uso
prompt = "quiero un producto para la garganta"

In [21]:

products = get_products(prompt)  # Llamar a la función para obtener productos
# Imprimir los productos obtenidos
print("Productos obtenidos:")
for product in products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos obtenidos:
Nombre: Clunia Maintenance Zn Gel, 59 ml, Descripción: Clunia Maintenance Zn Gel es un coloide para uso veterinario cuya aplicación principal está indicada para perros, gatos, reptiles, conejos, hurones, etc. Cuenta con una fórmula especialmente creada para ofrecer efectos antibacterianos en la cavidad bucal del animal a través del uso de ingredientes activos que crean una barrera efectiva y duradera contra microorganismos que causan el mal olor. Es un producto útil para el tratamiento a largo plazo de la gingivitis actuando profundamente en las encías sin resultar incómodo en su aplicación. Limpia el sarro en la cavidad bucodental, dejando los dientes limpios y con un aspecto saludable. Se adhiere fácilmente a la mucosa dental sin dejar marcas o sensaciones desagradables. Su composición está libre de elementos dañinos para la salud del animal. Controla adecuadamente las afecciones generadas por la halitosis. Deja una sensación de frescura en el área bucal. No pose

In [22]:
# Llamar a la función de reranking
ranked_products = rerank_products(prompt, products)["products"]  # Accede a la lista de productos

# Imprimir los productos rankeados
print("Productos rankeados:")
for product in ranked_products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos rankeados:
Nombre: Clunia Clinical ZnA Gel, 118 ml, Descripción: Clunia Clinical ZnA Gel es un gel diseñado para el cuidado y la limpieza de diversas heridas en la cavidad oral de perros y gatos. Posee características mucoadhesivas, lo que garantiza que el producto se mantenga por mayor tiempo en la zona afectada. Este gel no posee olor ni sabor, siendo además seguro en caso de ser ingerido por la mascota. Su acción y alivio son de rápido y prolongado efecto.Cuenta con un efecto antiséptico y cicatrizante, lo que ofrece un alivio sintomático del dolor y molestias relacionadas, ayudando a una ingesta normal de alimentos y líquidos. El gel viene contenido en un envase plástico de 118ml, contando con una punta fina para una adecuada y segura dosificación. Se puede aplicarse con ayuda de algodón o gasas suaves. El uso de este producto ayuda a la adecuada estimulación de la saliva, ayudando a la eliminación de diferentes bacterias y suciedad contenidas en la zona. Para mantener la

In [23]:
response_text = generate_response(prompt)  # Generar la respuesta
print(response_text)

{'type': 'product_search', 'message': 'He encontrado algunos productos que podrían ser útiles para la garganta:\n\n* **Clunia Clinical ZnA Gel, 118 ml:** Este gel contiene zinc y ácido hialurónico, ingredientes que ayudan a aliviar la irritación y la inflamación de la garganta. \n* **Clunia Maintenance Zn Gel, 59 ml:** Similar al anterior, este gel también contiene zinc y ácido hialurónico, ideal para el cuidado diario de la garganta.\n\n**Recomendaciones de uso:**\n\n* Aplicar el gel directamente sobre la garganta, realizando suaves masajes.\n* Se recomienda usar el gel varias veces al día, especialmente después de comer o beber.\n* Si los síntomas persisten, consulta con un profesional de la salud.\n\nRecuerda que estos productos no son un sustituto de un tratamiento médico. Si tienes alguna duda o preocupación, consulta con tu médico o farmacéutico. \n', 'products': [{'codigo_web': '013251', 'nombre': 'Clunia Clinical ZnA Gel, 118 ml', 'codigo_nacional': '0132513', 'descripcion': 'C